❯ 我现在想参考clevrer这个benchmark构建世界物理规律推断的benchmark，已经在此文件夹下实现了很少的初期想法。帮我想一下如何完善这个project。                   

● Explore(Explore Phy-bench project structure)
  ⎿  Done (15 tool uses · 65.9k tokens · 59s)
  (ctrl+o to expand)

● 现在我对整个项目有了清晰的了解，来给你一个系统的完善建议。

  Read 1 file (ctrl+o to expand)

● 现在我对整个项目有了完整了解。以下是系统性的完善建议：

  ---
  项目现状总结

  架构完整：10个世界配置、数据schema、评估协议、数据集构建器都写好了。
  核心缺口：
  1. 两个已有world脚本（black_hole、gravity_merry_go_round）是"旧版"，缺少run_episode()和states.jsonl导出
  2. 剩余8个world脚本未实现
  3. 评估pipeline只有协议定义，没有实际可运行的评测代码
  4. 与CLEVRER相比，任务类型过于单一

  ---
  完善方向（按优先级）

  1. 修复现有两个world脚本（最紧急）

  两个world脚本仍然是原型风格（直接渲染视频，无run_episode()），与dataset_builder.py的接口不兼容。需要：
  - 包装成标准run_episode(output_dir, seed, fps) -> dict
  - 添加StatesWriter导出每帧states.jsonl
  - 接入dataset_builder可调用

  2. 实现剩余8个world（最大工作量）

  按实现难度分层：

  ┌──────┬────────────────────────────────────────────────────────────────────┬──────────────────────────────┐
  │ 难度 │                               World                                │           关键挑战           │
  ├──────┼────────────────────────────────────────────────────────────────────┼──────────────────────────────┤
  │ 低   │ anti_gravity_bounce、bounce_accelerator                            │ 修改pymunk参数即可           │
  ├──────┼────────────────────────────────────────────────────────────────────┼──────────────────────────────┤
  │ 中   │ spring_floor、anti_newton_pendulum、gravity_merry_go_round（已有） │ 自定义collision handler      │
  ├──────┼────────────────────────────────────────────────────────────────────┼──────────────────────────────┤
  │ 高   │ time_dilation、mass_oscillation                                    │ 需要手动绕过pymunk的步进机制 │
  ├──────┼────────────────────────────────────────────────────────────────────┼──────────────────────────────┤
  │ 极高 │ collision_split、memory_metal_ball                                 │ 动态增减物体 + 状态快照      │
  └──────┴────────────────────────────────────────────────────────────────────┴──────────────────────────────┘

  3. 丰富任务类型（设计层面最重要的gap）

  参考CLEVRER的四类问题，当前Phy-Bench只有"轨迹预测"和"规律归纳"两类。建议增加：

  描述型（Descriptive）：
  - "视频中共发生了几次碰撞？"
  - "第3秒时物体的速度方向是？"

  解释型（Explanatory）：
  - "为什么小球没有落地而是向上运动？"（多选：a.有上升气流 b.重力向上 c.弹力过大）

  反事实型（Counterfactual）：
  - "如果在第2秒时移除中心黑洞，小球会做什么运动？"

  规律辨别型（新增，比CLEVRER更有特色）：
  - 给4段视频，3段来自同一world，1段来自另一world，找出异类
  - 给规律描述 + 多段视频，判断哪段符合该规律

  这些任务类型更适合直接用VLM（GPT-4o、Gemini）评测，不需要轨迹级别的预测能力。

  4. 评估pipeline实际化

  evaluator.py目前只是协议定义。需要：
  - LLM judge集成：调用API，用结构化prompt判断induction得分
  - 模型评测脚本：python evaluate.py --model gpt-4o --task induction
  - 基线模型：至少包含random baseline和一个VLM baseline（GPT-4o）

  5. 数据质量保障

  当前缺少：
  - 物理规律的可观测性验证：生成后自动检查关键事件是否出现（如anti_gravity_bounce必须有至少3次弹跳）
  - 视觉可辨性：物体颜色、标签、轨迹残影，让VLM更容易理解
  - 异常重试机制：生成失败或规律不明显时重新采样seed

  6. 数据集规模设计

  CLEVRER有约10k视频，但Phy-Bench是10个不同物理世界，每个world的episode数量和用途需要明确：

  每个world建议：
  ├── train/   30 episodes   # 用于induction任务
  ├── val/     10 episodes   # 开发调试
  └── test/    20 episodes   # 最终评测

  ---
  建议的实施顺序

  Phase 1（可运行原型）
    └─ 修复 black_hole + gravity_merry_go_round → 接入 dataset_builder → 能生成数据

  Phase 2（完整世界）
    └─ 实现剩余8个world（从低难度到高难度）

  Phase 3（任务扩展）
    └─ 增加QA任务类型 → 设计问题模板 → 生成问题-答案对

  Phase 4（评测pipeline）
    └─ LLM judge实现 → 基线模型评测 → 结果分析脚本

  ---
  你现在想先从哪个方向开始？ 我可以：
  - A. 帮你把现有的两个world脚本重构成符合接口的版本（立刻可运行）
  - B. 帮你设计并实现某几个world的仿真脚本
  - C. 帮你在任务类型层面做benchmark设计的扩展

✻ Worked for 2m 2s

❯ 我目前这两个世界仅是一个例子，实际存在的世界也不仅仅会有10个，可能是一系列的世界的案例。现在最关键的是基于目前的技术栈（pybullet，pymunk等）最终应该实现 
怎样的数据形式，有哪些任务和评价指标。以及一些实际场景的价值（这个benchmark怎么阐释它自身的定位）。                                                        

● 这是个需要认真想清楚的设计问题，我来系统性地分析。

  ---
  一、基准的核心定位

  这个benchmark在解决什么问题？

  现有物理推理benchmark（CLEVRER、PhysicsIQ等）都有一个共同假设：物理规律是已知的，测的是模型能否在已知物理定律下做因果/预测推理。

  Phy-Bench的本质不同在于：物理规律本身是未知的，需要从观察中归纳。

  这对应了一个更基础的认知能力——归纳泛化（inductive generalization）：在不知道"这个世界遵循什么规则"的前提下，从少量视频中抽象出规律，并将其迁移到新场景。

  已有benchmark：模型 + 已知规律 → 推理特定事件
  Phy-Bench：   模型 + 未知世界视频 → 归纳规律 → 推理特定事件

  实际场景的价值论述

  机器人/具身智能：机器人部署在新环境时，地面摩擦、物体弹性、工具物理属性都是未知的，需要通过少量交互快速建立"这个环境的物理模型"。Phy-Bench测的正是这种"快
  速物理模型归纳"能力。

  科学发现自动化：在实验室模拟中，AI需要从观测数据推断未知物理参数或规律（如新材料的力学特性）。

  世界模型的真实评估：当前世界模型（Genie、DIAMOND等）的评估依赖感知质量指标（FID、LPIPS），无法衡量是否真正理解物理因果结构。Phy-Bench提供了一个直接评估"
  物理因果理解"的探针。

  一句话定位："Phy-Bench evaluates whether models can act as physicists in unknown universes — not just predict outcomes, but induce the laws that govern
  them."

  ---
  二、数据形式

  核心设计原则

  1. 视频是一等公民：模型输入是视频，state序列是评测ground truth，两者分离
  2. 世界无关的schema：同一套格式同时支持pymunk（2D）和pybullet（3D）
  3. 事件日志是关键中间层：QA对从事件日志自动生成，而非人工标注

  完整数据结构

  data/
  └── worlds/
      └── {world_id}/
          ├── world_meta.json          # 世界元信息（规律定义对外隐藏）
          ├── rule_description.json    # 多粒度自然语言描述（oracle用）
          └── episodes/
              └── ep_{NNN}/
                  ├── video.mp4        # 主要模型输入（RGB，30fps）
                  ├── frames/          # 可选：单帧PNG（for图像模型）
                  ├── states.jsonl     # 每帧物理状态（评测ground truth）
                  ├── events.jsonl     # 离散事件序列（QA生成的来源）
                  ├── qa_pairs.json    # 结构化问答对
                  └── episode_meta.json

  states.jsonl 字段设计

  {
    "frame": 42,
    "timestamp": 1.40,
    "objects": [
      {
        "id": 0,
        "type": "ball",
        "position": [320.5, 240.1],      // 2D pymunk / [x,y,z] pybullet
        "velocity": [-50.2, 120.0],
        "orientation": 0.0,              // 朝向角（弧度）
        "angular_velocity": 0.3,
        "mass": 1.0,                     // 质量震荡世界中会变化
        "radius": 20.0,                  // 碰撞分裂世界中会变化
        "properties": {}                 // 世界特有属性的扩展槽
      }
    ],
    "world_state": {
      "gravity": [0.0, -500.0],         // 重力向量
      "time_scale": 1.0,                // 时间减速带世界
      "custom": {}                      // 世界特有全局状态
    }
  }

  关键决策：properties和custom是扩展槽，让schema对新世界保持前向兼容，不需要每加一个世界就改schema。

  events.jsonl 字段设计

  {"timestamp": 2.13, "type": "collision",    "obj_ids": [0, 1], "data": {"impact_speed": 230.5}}
  {"timestamp": 3.40, "type": "bounce",       "obj_ids": [0],    "data": {"surface": "floor", "pre_speed": 150.0, "post_speed": 300.0}}
  {"timestamp": 5.00, "type": "gravity_turn", "obj_ids": [],     "data": {"direction": "right"}}
  {"timestamp": 5.00, "type": "teleport",     "obj_ids": [1],    "data": {"from": [100,200], "to": [340, 180]}}
  {"timestamp": 6.21, "type": "split",        "obj_ids": [2],    "data": {"children": [4, 5]}}
  {"timestamp": 7.00, "type": "mass_peak",    "obj_ids": [0],    "data": {"mass": 2.0, "phase": "max"}}

  事件日志是QA自动生成的唯一来源，设计好这个，QA就是确定性程序生成的，无需人工标注。

  ---
  三、任务设计

  任务分类矩阵

  ┌─────────────────┬─────────────────────────────┬───────────┬──────────────────────┐
  │    任务类型     │            输入             │ 输出形式  │       评测方式       │
  ├─────────────────┼─────────────────────────────┼───────────┼──────────────────────┤
  │ T1 规律归纳     │ K段训练视频                 │ 自由文本  │ LLM judge            │
  ├─────────────────┼─────────────────────────────┼───────────┼──────────────────────┤
  │ T2 规律辨别     │ 4段视频，选异类             │ 单选      │ Accuracy             │
  ├─────────────────┼─────────────────────────────┼───────────┼──────────────────────┤
  │ T3 事件预测     │ 视频前N帧 + 问题            │ 单选/是否 │ Accuracy             │
  ├─────────────────┼─────────────────────────────┼───────────┼──────────────────────┤
  │ T4 规律条件预测 │ 规律描述 + 视频前N帧 + 问题 │ 单选      │ Accuracy             │
  ├─────────────────┼─────────────────────────────┼───────────┼──────────────────────┤
  │ T5 反事实推断   │ 视频 + 假设性问题           │ 单选      │ Accuracy             │
  ├─────────────────┼─────────────────────────────┼───────────┼──────────────────────┤
  │ T6 轨迹预测     │ 视频前N帧 + 状态序列        │ 坐标序列  │ ADE/FDE + 物理一致性 │
  └─────────────────┴─────────────────────────────┴───────────┴──────────────────────┘

  重要：T6（轨迹预测）仍然保留，但不是VLM的主要任务，而是专门给具有状态输入/输出能力的world model（如Dreamer、UniSim）用的。VLM走T1-T5。

  QA模板举例

  T3 事件预测（自动生成）：
  events.jsonl中有一次bounce事件，pre_speed=150, post_speed=300
  → 生成问题："小球以150像素/秒的速度撞墙，弹出速度是多少？"
     选项：a) 75  b) 150  c) 300  d) 450
     答案：c（需要在视频中归纳出"弹跳加速"规律才能答对）

  T5 反事实推断（模板生成）：
  "如果这个世界中不存在重力方向旋转，第8秒时球会在屏幕的哪个区域？"
  a) 左下  b) 右下  c) 左上  d) 右上

  T2 规律辨别：
  给4段视频，其中3段来自"反弹加速器"，1段来自"正常弹性碰撞"
  问："哪段视频中的物理规律与其他三段不同？"

  ---
  四、评价指标

  T1 规律归纳（核心指标）

  用结构化的LLM judge，不是端到端打分，而是分解成可验证的子项：

  归纳得分 =
    40% × 必要事实覆盖率（关键物理现象是否提到）
    15% × 精确度奖励（数值是否准确，如"速度翻倍"而非"速度增加"）
    15% × 零幻觉奖励（是否没有错误描述Newtonian-违背的现象）
    30% × T4迁移得分（用归纳出的规律去做条件预测，作为归纳质量的行为测试）

  最后一项（30%）是关键设计：用预测任务验证归纳质量，避免归纳文本"看起来对但实际没用"的问题。

  T2-T5 多选题

  - 单世界准确率：每个world的平均正确率
  - 规律相关题 vs 规律无关题：分别统计（诊断模型是否真正理解了规律）
  - 零样本 vs 少样本（K=1,3,5段训练视频）：学习效率曲线

  T6 轨迹预测（针对world model）

  ADE（平均位移误差）  = mean(||pred_pos - gt_pos||) over all frames & objects
  FDE（最终位移误差）  = ||pred_pos_final - gt_pos_final||
  物理一致性得分       = 世界特有规则的满足率（如弹跳后速度是否翻倍）

  跨任务的综合指标

  Phy-Score = 0.4 × T1_induction + 0.3 × T3/T4_avg_accuracy + 0.3 × T5_counterfactual

  T2规律辨别单独报告——它主要衡量感知层面的规律识别，而非推理。

  ---
  五、评测设计的关键维度

  难度轴（三个维度）

  1. 规律显著性：规律是否在短时间内可见（5秒够看出来 vs 需要30秒）
  2. 规律复合度：单一规律（反重力）vs 复合规律（质量震荡 + 碰撞分裂同时发生）
  3. 干扰强度：视频中有多少"正常物理"事件在分散注意力

  消融设计

  ┌──────────────┬──────────────────────────┬────────────────────────┐
  │    变体名    │           说明           │          目的          │
  ├──────────────┼──────────────────────────┼────────────────────────┤
  │ oracle-rule  │ 给模型提供规律的文字描述 │ 测应用能力上限         │
  ├──────────────┼──────────────────────────┼────────────────────────┤
  │ oracle-state │ 给模型提供states.jsonl   │ 剥离视觉感知，测纯推理 │
  ├──────────────┼──────────────────────────┼────────────────────────┤
  │ video-only   │ 只给视频                 │ 端到端基线             │
  ├──────────────┼──────────────────────────┼────────────────────────┤
  │ K=1/3/5      │ 不同数量训练视频         │ 样本效率               │
  └──────────────┴──────────────────────────┴────────────────────────┘

  oracle-state和video-only的得分差 = 视觉感知对该world的难度；oracle-rule和video-only的得分差 = 规律归纳的难度。这两个gap是非常有价值的诊断信息。

  ---
  六、技术栈建议

  pymunk（2D）：适合绝大多数当前world（弹跳、重力、碰撞类），实现简单，渲染快。继续作为主力。

  pybullet（3D）：为后续扩展保留，3D world可以测试"从2D投影归纳3D规律"这类更难的任务。暂不是优先级。

  视频渲染：建议在当前pygame基础上统一：
  - 分辨率：640×480（足够清晰，不太大）
  - FPS：30
  - 物体：不同颜色圆形（颜色固定，split后子球继承颜色），加ID标注可选
  - 轨迹残影（可选层）：最近0.5秒的轨迹点，帮助VLM理解运动趋势

  ---
  现在你需要做的最关键决策：T3-T5的多选题是程序自动生成还是人工/LLM辅助生成？前者快但覆盖有限，后者质量高但成本大。我的建议是：用events.jsonl做程序生成的核
  心骨架，用LLM做问题表述的改写，做到"内容确定性 + 表达多样性"。

✻ Cogitated for 1m 48s

❯ 我有一个设想是，先做二维场景下的。具体的场景是一些三维情况下被遮挡或者容易产生视觉错觉的情形之下的二维观测视频，能否推断出具体的物理情景和位置关系是怎样 
的，当然这是一个粗略的描述，还有其他可细化的方向。帮我完善这一构想。                                                                                       

● 这是一个比"虚构物理规则"更有深度的方向，让我来系统地展开它。

  ---
  核心问题的重新表述

  你的构想可以精确化为：

  ▎ 给定一段二维视频观测，模型能否推断出产生这一观测的三维物理真实？

  这里的关键张力是：同一段2D视频，可能对应多种不同的3D物理场景。测的不是"模型能否感知视觉内容"，而是"模型能否从欠定的2D投影中，通过物理推理消除歧义、恢复真
  实"。

  这与现有benchmark的本质区别：
  - CLEVRER等：已知物理规律 + 完整可见场景 → 因果推理
  - 你的方向：Newtonian物理已知，但观测本身是有损的 → 需要物理推理来补全信息

  ---
  一、2D观测歧义的分类体系

  这是整个benchmark的核心结构，应该沿着"歧义类型"来组织世界/场景：

  类型A：深度歧义（Depth Ambiguity）

  2D投影丢失了z轴信息，同样的2D轨迹可以对应截然不同的3D运动。

  场景A1 — 轨道/振荡歧义
  一个质点在2D上做来回直线运动。实际可能是：
  - 一维弹簧振荡
  - 正圆轨道从侧面看（投影成线段）
  - 椭圆轨道以特定角度俯视

  三种情况的能量、力的大小完全不同。

  场景A2 — 抛物线/匀速歧义
  相机倾斜45°拍摄一个抛体运动，2D轨迹看起来像匀速直线运动（抛物线的侧投影恰好是直线）。问：这是自由飞行还是受力运动？

  场景A3 — 接近/旋转歧义
  物体在2D上看起来越来越大。可能是：
  - 物体向相机靠近
  - 物体本身在膨胀
  - 球体在旋转（如果有表面纹理）

  ---
  类型B：遮挡歧义（Occlusion Ambiguity）

  场景B1 — 幽灵碰撞（Phantom Collision）
  两个球的2D投影发生重叠，然后各自继续运动。实际上可能是：
  - 真实碰撞（同深度）→ 速度交换
  - 深度不同擦肩而过 → 各自保持速度
  - 一个从另一个后方通过（完全遮挡后出现）

  通过碰撞前后的速度变化可以推断，但需要模型理解物理守恒定律。

  场景B2 — 遮挡体后的运动
  球进入一个不透明区域（如一个盒子的后方），从另一侧出现，轨迹改变了。问：遮挡区域内发生了什么？可能是：
  - 撞到了遮挡体内的隐藏障碍
  - 遮挡体内有斜坡改变了方向
  - 完全不同速度的另一个球出来了（调包问题）

  场景B3 — 物体计数
  视频中有几个物体？部分时间某些物体被其他物体完全遮挡。这测试了"物体永久性（object permanence）"与物理连续性的结合推理。

  ---
  类型C：透视/尺度歧义（Perspective Ambiguity）

  场景C1 — 尺寸-距离权衡
  两个球在2D上大小不同。可能是：
  - 相同大小，不同深度
  - 不同大小，相同深度

  当它们发生碰撞时，碰撞后的速度变化会揭示真实质量比（假设密度相同），从而可以推断谁更大/谁更近。

  场景C2 — 相机倾斜重力方向推断
  重力在2D中显示为某个角度方向的加速度。问：是相机倾斜了，还是这个世界的重力方向本来就是斜的？（可以通过分析多个独立物体的运动是否共享同一"重力方向"来推断
  。）

  ---
  类型D：因果来源歧义（Causal Source Ambiguity）

  场景D1 — 隐藏力源
  球在空旷场景中突然改变加速度。可能是：
  - 画面外有一个磁铁（不可见）
  - 球到达了某个隐藏边界并弹回
  - 另一个与背景同色的隐藏球发生了碰撞

  场景D2 — 主动vs被动运动
  球沿直线匀速运动。可能是：
  - 完全没有摩擦的自由滑行
  - 有一个持续推力恰好抵消摩擦力
  - 在一个隐藏的轨道/槽中运动（实际上是被约束的）

  区分这两种情况需要观察它与其他物体碰撞时的行为。

  ---
  二、任务设计

  基于以上分类，任务结构应该是：

  核心任务：多假说鉴别（Hypothesis Discrimination）

  每个场景提供一段2D视频，给出2-4个候选的3D物理解释，要求选择最符合观测的那个。

  视频：两球2D轨迹发生重叠，之后速度明显变化
  选项：
    A. 同深度正面碰撞（动量交换）
    B. 不同深度擦肩而过（互不影响）
    C. 一球从另一球正后方穿过（遮挡）
    D. 磁性斥力使两球减速

  这是可精确评测（多选正确率）且对VLM友好的任务形式。

  扩展任务

  ┌──────────────────┬──────────────────────────────────────────┬──────────────────────────────┬───────────────┐
  │       任务       │                   输入                   │             输出             │     意义      │
  ├──────────────────┼──────────────────────────────────────────┼──────────────────────────────┼───────────────┤
  │ 3D配置还原       │ 2D视频                                   │ 哪个3D示意图与之一致         │ 逆图形学推理  │
  ├──────────────────┼──────────────────────────────────────────┼──────────────────────────────┼───────────────┤
  │ 物体计数         │ 含遮挡的视频                             │ 场景中共有几个物体           │ 物体永久性    │
  ├──────────────────┼──────────────────────────────────────────┼──────────────────────────────┼───────────────┤
  │ 碰撞真实性判断   │ 包含视觉碰撞的视频                       │ 是否发生了真实物理碰撞       │ 深度感知+物理 │
  ├──────────────────┼──────────────────────────────────────────┼──────────────────────────────┼───────────────┤
  │ 反事实预测       │ 视频前半段 + 假设（"如果相机视角转90°"） │ 从新视角看到的画面描述       │ 3D心理模型    │
  ├──────────────────┼──────────────────────────────────────────┼──────────────────────────────┼───────────────┤
  │ 遮挡区域内容推断 │ 含遮挡段的视频                           │ 遮挡区域发生了什么           │ 物理外推      │
  ├──────────────────┼──────────────────────────────────────────┼──────────────────────────────┼───────────────┤
  │ 力场来源定位     │ 物体异常运动的视频                       │ 力源位置（在画面的哪个方向） │ 逆动力学      │
  └──────────────────┴──────────────────────────────────────────┴──────────────────────────────┴───────────────┘

  ---
  三、数据生成策略

  关键设计：真实物理发生在3D，只把2D投影给模型。

  3D物理引擎（pybullet / 手写简化3D）
      ↓
  投影层（相机内外参，定义哪些物体可见）
      ↓
  遮挡渲染层（哪些物体被哪些物体挡住）
      ↓
  2D视频输出（模型输入）

  3D状态序列保留为 ground truth（评测用）

  近期可行方案（不引入完整3D引擎）：

  用pymunk做2D物理，但增加一个"伪深度层"：
  - 每个物体有一个 z 属性（不参与物理计算）
  - 渲染时按 z 值缩放大小（近大远小）
  - z 值不同的物体即使2D位置重叠也不物理碰撞
  - 不透明遮挡物只是一个渲染层，物理世界里不存在

  这样可以快速生成大量覆盖A、B、C类歧义的场景，不需要真正的3D物理引擎。

  ---
  四、benchmark的定位与价值

  填补的认知科学-AI交叉空白

  这个benchmark直接对应认知科学中的"朴素物理（intuitive physics）"研究——人类婴儿从6个月起就会对"物体永久性"、"不可穿透性"有预期，对"幽灵碰撞"会表现出惊讶。
  测试AI是否真正建立了这种3D物理直觉，而不仅仅是拟合了2D视频的统计规律。

  对现有视觉模型的诊断价值

  当前VLM（GPT-4V、Gemini等）大量训练于2D图像/视频，缺乏3D物理先验的归纳。这个benchmark可以回答：

  ▎ "模型的视觉理解是真正的3D物理推理，还是2D模式匹配的幻象？"

  具体可以设计对照实验：
  - 人类表现参照：同样场景，人类通常能正确判断（这是视觉系统的进化能力）
  - 模型系统性错误分析：模型倾向于把深度不同的擦肩误判为碰撞吗？倾向于低估遮挡物体数量吗？

  实际应用锚点

  - 机器人抓取：相机拍到两个物体"重叠"——它们是真的叠在一起还是一前一后？这直接影响抓取规划
  - 自动驾驶：行人被车辆遮挡后出现在另一侧——是同一个人还是两个人？
  - 视频监控分析：物体碰撞检测——2D画面上的接触是否对应真实的物理碰撞？
  - 科学视频分析：显微镜视频中，物体运动轨迹看起来异常——是真实的物理现象还是焦平面以外的物体投影干扰？

  ---
  五、与原有Phy-Bench的关系

  这两个方向实际上可以构成一个benchmark的两个轨道：

  Phy-Bench
  ├── Track A：规律归纳（Law Induction）
  │   └── 完整可观测的2D世界，但物理规律是非牛顿的
  │       测：能否从观测归纳出未知规律？
  │
  └── Track B：场景还原（Scene Reconstruction）  ← 你的新想法
      └── 牛顿物理已知，但2D观测有信息损失
          测：能否从欠定的2D投影还原真实的3D物理场景？

  两个track测的是互补的能力，合在一起才是"完整的物理世界理解"：既能归纳规律，又能穿透感知歧义还原真实。

  ---
  接下来最关键的一步：需要确定Track B的"核心场景集"——哪5-8类场景是最有区分力、最有物理意义、且技术上可以用pymunk+伪深度层实现的？我可以帮你逐一设计这些具体
  场景，或者先把最典型的"幽灵碰撞"和"遮挡轨迹推断"场景用代码实现出来验证可行性。

✻ Worked for 1m 58s

❯ 有什么具体的工程实践建议                                                                                                                                 

● Explore(Read all existing code files)
  ⎿  Done (5 tool uses · 36.2k tokens · 20s)
  (ctrl+o to expand)

● 现在基于具体代码给出工程建议，分成几个层次：

  ---
  一、最关键的架构决策

  当前代码的问题是把物理模拟和渲染耦合在同一个脚本里（两个world脚本都是pygame event loop直接跑的）。新方向需要先把这两者分离，再加入第三层——投影/遮挡层。

  当前：[pymunk物理] + [pygame渲染] → video.mp4  （单脚本，耦合）

  目标：[pymunk物理] → [3D状态] → [投影层] → [pygame渲染] → video.mp4
                             ↓
                        [GT导出] → states_3d.jsonl + events.jsonl + qa_pairs.json

  ---
  二、伪3D层的具体实现

  用pymunk做2D物理，但每个物体多一个z属性。物理计算完全忽略z，z只影响渲染。

  # worlds/core/renderer.py

  import math
  from dataclasses import dataclass, field
  from typing import List, Tuple
  import pygame

  FOCAL_LENGTH = 600.0   # 透视焦距，控制近大远小的强度
  CANVAS_CENTER = (320, 240)

  @dataclass
  class Object3D:
      pymunk_body: object          # pymunk Body，持有2D物理状态
      z: float = 0.0               # 深度：0=最近，正值=更远
      base_radius: float = 20.0
      color: Tuple[int,int,int] = (200, 80, 80)
      obj_id: int = 0

      @property
      def screen_pos(self) -> Tuple[float, float]:
          """带透视偏移的屏幕坐标：远处物体向画面中心收缩"""
          px, py = self.pymunk_body.position
          cx, cy = CANVAS_CENTER
          scale = FOCAL_LENGTH / (FOCAL_LENGTH + self.z)
          sx = cx + (px - cx) * scale
          sy = cy + (py - cy) * scale
          return sx, sy

      @property
      def screen_radius(self) -> float:
          return self.base_radius * FOCAL_LENGTH / (FOCAL_LENGTH + self.z)


  def render_frame(surface: pygame.Surface, objects: List[Object3D],
                   occluders=None, depth_cue=False):
      """
      按z值从大到小排序绘制（painter's algorithm：远的先画）
      occluders: 不参与物理、只遮挡视线的静态区域列表
      depth_cue: 是否通过颜色饱和度暗示深度（easy模式开启）
      """
      surface.fill((30, 30, 30))

      # 先画遮挡物（作为背景层）
      if occluders:
          for occ in occluders:
              pygame.draw.rect(surface, (60, 60, 60), occ.rect)

      # 按深度从远到近排序
      sorted_objs = sorted(objects, key=lambda o: -o.z)
      for obj in sorted_objs:
          pos = obj.screen_pos
          r = obj.screen_radius
          color = obj.color
          if depth_cue:
              # 远处物体颜色更暗（可选，给easy模式用）
              fade = FOCAL_LENGTH / (FOCAL_LENGTH + obj.z)
              color = tuple(int(c * fade) for c in color)
          pygame.draw.circle(surface, color, (int(pos[0]), int(pos[1])), max(2, int(r)))

  关键参数：
  - FOCAL_LENGTH大 → 近大远小效果弱（更难判断深度）
  - FOCAL_LENGTH小 → 深度线索明显（更容易）
  - 这直接对应难度控制

  ---
  三、场景配置系统

  替换现有的WorldConfig，改为声明式的SceneConfig，每类歧义场景对应一个SceneType：

  # benchmark/scene_configs.py

  from dataclasses import dataclass, field
  from enum import Enum
  from typing import List, Dict, Any

  class AmbiguityType(Enum):
      DEPTH_PHANTOM_COLLISION = "depth_phantom_collision"   # 幽灵碰撞
      DEPTH_ORBIT_VS_OSCILLATION = "depth_orbit_oscillation"  # 轨道/振荡歧义
      OCCLUSION_TRAJECTORY = "occlusion_trajectory"          # 遮挡轨迹
      OCCLUSION_COUNT = "occlusion_count"                    # 物体计数
      PERSPECTIVE_SIZE_DISTANCE = "perspective_size_distance" # 尺寸-距离权衡
      CAUSAL_HIDDEN_FORCE = "causal_hidden_force"             # 隐藏力源

  @dataclass
  class SceneConfig:
      scene_type: AmbiguityType
      # 物理参数
      num_objects: int = 4
      duration: float = 8.0
      fps: int = 30
      resolution: tuple = (640, 480)
      # 歧义控制参数
      depth_separation: float = 150.0   # 深度差（像素等效）
      occlusion_coverage: float = 0.0   # 遮挡区域占画面比例
      camera_tilt_deg: float = 0.0      # 相机倾斜角度
      focal_length: float = 600.0       # 影响近大远小强度
      # 问答对配置
      qa_templates: List[str] = field(default_factory=list)
      # 元信息
      difficulty: str = "medium"  # easy / medium / hard
      tags: List[str] = field(default_factory=list)

  # 预定义场景
  PHANTOM_COLLISION_EASY = SceneConfig(
      scene_type=AmbiguityType.DEPTH_PHANTOM_COLLISION,
      depth_separation=250.0,   # 深度差很大，2D重叠但3D距离远
      focal_length=400.0,       # 明显的近大远小
      difficulty="easy",
  )
  PHANTOM_COLLISION_HARD = SceneConfig(
      scene_type=AmbiguityType.DEPTH_PHANTOM_COLLISION,
      depth_separation=50.0,    # 深度差小，视觉上几乎无法区分
      focal_length=1200.0,      # 透视效果很弱
      difficulty="hard",
  )

  ---
  四、Ground Truth Schema扩展

  在现有data_schema.py基础上，扩展以支持3D信息和视觉事件：

  # 在 data_schema.py 中新增

  @dataclass
  class Object3DState:
      id: int
      position_2d: tuple      # pymunk计算的2D位置
      z: float                # 深度（GT，不给模型看）
      position_screen: tuple  # 渲染后的屏幕坐标
      velocity_2d: tuple
      radius_world: float     # 实际半径
      radius_screen: float    # 渲染后的屏幕半径

  @dataclass
  class VisualEvent:
      """2D观测中出现的视觉现象——这是模型能看到的"""
      frame: int
      timestamp: float
      type: str   # "2d_overlap" | "disappear_into_occluder" | "reappear_from_occluder"
      obj_ids: list

  @dataclass
  class PhysicsEvent:
      """3D世界中真实发生的物理事件——这是GT"""
      frame: int
      timestamp: float
      type: str   # "real_collision" | "no_collision_depth_diff" | "hidden_bounce"
      obj_ids: list
      data: dict  # 速度变化、深度差等

  states_3d.jsonl（GT，评测用）和states_2d.jsonl（只含屏幕坐标，模型可选用）分开存储。

  ---
  五、QA自动生成

  这是最关键的工程投入点。从PhysicsEvent自动生成结构化QA：

  # benchmark/qa_generator.py

  from typing import List
  import random

  class QAGenerator:
      def generate(self, scene_config, physics_events, visual_events, rng) -> List[dict]:
          generators = {
              AmbiguityType.DEPTH_PHANTOM_COLLISION: self._phantom_collision_qa,
              AmbiguityType.OCCLUSION_COUNT: self._object_count_qa,
              AmbiguityType.OCCLUSION_TRAJECTORY: self._occlusion_trajectory_qa,
          }
          return generators[scene_config.scene_type](
              scene_config, physics_events, visual_events, rng
          )

      def _phantom_collision_qa(self, cfg, phys_events, vis_events, rng):
          overlaps = [e for e in vis_events if e.type == "2d_overlap"]
          real_collisions = {
              (e.obj_ids[0], e.obj_ids[1])
              for e in phys_events if e.type == "real_collision"
          }

          qa_pairs = []
          for overlap in overlaps:
              oids = tuple(sorted(overlap.obj_ids))
              actually_collided = oids in real_collisions

              qa_pairs.append({
                  "id": f"phantom_{overlap.frame}",
                  "type": "collision_judgment",
                  "question": f"在第{overlap.frame}帧附近，球{oids[0]}和球{oids[1]}的轨迹在画面中发生了重叠。请问它们是否发生了真实的物理碰撞？",
                  "choices": [
                      "是，发生了碰撞，之后速度发生了动量交换",
                      "否，两球深度不同，只是投影重叠，互不影响"
                  ],
                  "answer": 0 if actually_collided else 1,
                  "difficulty": cfg.difficulty,
                  "evidence_frames": [overlap.frame - 5, overlap.frame + 5],
                  # 给评测脚本用的元信息
                  "meta": {
                      "real_collision": actually_collided,
                      "depth_separation": cfg.depth_separation,
                  }
              })
          return qa_pairs

      def _object_count_qa(self, cfg, phys_events, vis_events, rng):
          # 在遮挡最严重的帧问：当前画面中可见的球有几个？场景中总共有几个球？
          ...

  核心原则：问题的正确答案是从PhysicsEvent确定性计算出的，不存在模糊性。distractors要具有物理合理性。

  ---
  六、评测harness

  # evaluate.py

  import anthropic
  import base64
  import json
  from pathlib import Path

  def evaluate_episode(episode_dir: Path, model: str = "claude-opus-4-6") -> dict:
      client = anthropic.Anthropic()
      qa_pairs = json.loads((episode_dir / "qa_pairs.json").read_text())

      # 将视频关键帧抽取为图片序列（VLM接受图片，不是视频）
      frames = extract_key_frames(episode_dir / "video.mp4", n=8)

      results = []
      for qa in qa_pairs:
          prompt = build_prompt(qa, frames)
          response = client.messages.create(
              model=model,
              max_tokens=256,
              messages=[{"role": "user", "content": prompt}]
          )
          pred = parse_choice(response.content[0].text)
          results.append({
              "qa_id": qa["id"],
              "correct": pred == qa["answer"],
              "predicted": pred,
              "ground_truth": qa["answer"],
              "difficulty": qa["difficulty"],
              "type": qa["type"],
          })
      return results


  def build_prompt(qa: dict, frames: list) -> list:
      """多帧图片 + 问题文本 → Claude message content"""
      content = []
      for i, frame_b64 in enumerate(frames):
          content.append({
              "type": "image",
              "source": {"type": "base64", "media_type": "image/png", "data": frame_b64}
          })
      content.append({
          "type": "text",
          "text": f"{qa['question']}\n\n" +
                  "\n".join(f"{i+1}. {c}" for i, c in enumerate(qa['choices'])) +
                  "\n\n请只回答选项编号（1或2）。"
      })
      return content

  ---
  七、目录结构建议

  Phy-bench/
  ├── benchmark/
  │   ├── scene_configs.py     # SceneConfig + AmbiguityType（替换world_configs）
  │   ├── data_schema.py       # 扩展3D state + visual/physics events
  │   ├── qa_generator.py      # QA自动生成（按scene_type dispatch）
  │   ├── dataset_builder.py   # 不变，调orchestration逻辑
  │   └── evaluator.py         # 统计accuracy，按type/difficulty分组
  ├── worlds/
  │   ├── core/
  │   │   ├── renderer.py      # Object3D + render_frame（透视层）
  │   │   ├── physics.py       # pymunk封装 + 事件检测
  │   │   └── video_writer.py  # opencv录制
  │   ├── phantom_collision.py
  │   ├── occlusion_trajectory.py
  │   └── ...
  ├── evaluate.py              # CLI评测入口
  └── data/
      └── scenes/
          └── {scene_type}/
              └── {difficulty}/
                  └── ep_{NNN}/
                      ├── video.mp4
                      ├── states_2d.jsonl    # 屏幕坐标（模型可用）
                      ├── states_3d.jsonl    # 含深度的GT（评测用）
                      ├── events.jsonl       # visual + physics events
                      └── qa_pairs.json

  ---
  八、立即可以开始的第一步

  按工程依赖顺序：

  Step 1：实现worlds/core/renderer.py中的Object3D和render_frame，用一个hardcode脚本验证透视渲染效果是否符合预期（两个深度不同的球在屏幕上大小不同、靠近中心
  的偏移是否正确）。

  Step 2：实现phantom_collision.py——这是最简单的场景（只需要控制两个球的z值，让它们2D轨迹相交但物理上不碰撞），验证整个pipeline：物理模拟→渲染→states导出→Q
  A生成→视频。

  Step 3：把QA生成跑通，哪怕只有一种题型，确认qa_pairs.json格式正确，evaluate.py可以调用Claude API打分。

  有了这三步，整个benchmark的主干就是确定的了，后续加新场景类型只需要实现新的SceneConfig和对应的QAGenerator方法。

  ---
  一个需要提前想清楚的问题：视频给VLM时用连续帧图片序列还是视频文件？目前Claude
  API支持图片但不支持视频文件，所以evaluate.py需要把视频抽帧。建议每个episode同时保存8-16张关键帧PNG，避免每次评测都要解码视频。